# HW 6 — H5: obdurate (committed) splits

**My predictions (Oct 25, 2026):**
- Committed splits occur; on Full they take about 10% of round time.
- The median duration of a committed split is 5–20 s.
- The player who breaks a committed split is in the minority more often than chance (share above k/7).

Data: `load_runs("outcome", roles={"train"})`: train groups only, rounds with no bot and with an end marker (aborted 1-row rounds and unplayed waves are dropped). Topologies Full, Hierarchy, Star. Val is not used.

In [1]:
import pandas as pd
from slider import config
from slider import slider_logs as sl
from slider import h5

runs, table = sl.load_runs("outcome", roles={"train"})
pd.crosstab(table["dataset"], table["topology"], margins=True)

topology,Full,Hierarchy,Star,All
dataset,,,,
CC,31,34,33,98
CL,42,41,0,83
EL,0,33,0,33
All,73,108,33,214


## 1. Committed splits

A row is a committed split if the signs are mixed and every |s_i| ≥ 0.9. Spells use the forward hold and merge gaps shorter than τ = 1 s (`spells` in `src/slider/fraught.py`). Mean % committed is the share of each round spent in committed rows, averaged over rounds. KM median: escape = the spell ended; censored = still open when a failed round ended.

In [2]:
sp, per_run = h5.committed_spells(runs, table, cutoff=0.9, tau=1.0)
h5.topology_summary(sp, per_run).round(2)

,runs,mean % committed,spells,KM median (s),consensus %,relapse %,unresolved %,censored %
topology,,,,,,,,
Full,73,9.74,159,2.36,27.04,71.07,0.63,1.26
Hierarchy,108,11.04,340,1.54,18.24,80.29,1.18,0.29
Star,33,9.34,35,2.61,54.29,45.71,0.00,0.00


Full only, by k = minority size at the spell's last committed row.

In [3]:
full = sp[sp["topology"] == "Full"]
h5.spell_summary(full, "k", [1, 2, 3]).round(2)

,spells,KM median (s),consensus %,relapse %,unresolved %,censored %
k,,,,,,
1,74,5.00,32.43,66.22,0.00,1.35
2,53,1.32,28.30,67.92,1.89,1.89
3,32,0.94,12.50,87.50,0.00,0.00


## 2. Who breaks a committed split (Full)

Breaker = the player whose |s| drops below 0.9 in the row after the spell (spells still open at the end of a failed round are left out). If several players drop in that row the spell is ambiguous and left out. Null: the breaker is a random player, so P(minority) = k/7. 95% CI of (observed − expected) / n: 10,000 bootstrap resamples of groups (log files), seed 573.

In [4]:
pd.Series(h5.breaker_summary(full)).round(3).to_frame("Full")

,Full
spells ended,157.000
ambiguous,2.000
no drop,0.000
breaker spells (n),155.000
groups,24.000
observed minority,60.000
expected minority (sum k/7),38.571
observed share,0.387
expected share,0.249
(obs - exp) / n,0.138


## 3. Confound: was the breaker the last player to reach |s| ≥ 0.9 before the spell started? (null 1/7)

In [5]:
pd.Series(h5.confound_summary(full)).round(3).to_frame("Full")

,Full
breaker spells with a clear last reacher (n),153.000
breaker = last to reach,28.000
share,0.183
null share,0.143
ambiguous last reacher,2.000


## 4. Sensitivity: cutoff 0.8 / 0.9 / 1.0 × τ 0.5 / 1 / 2 s

Saved to `results/tables/h5_train.csv`. Mean % committed does not depend on τ.

In [6]:
grid = h5.sensitivity(runs, table, cutoffs=[0.8, 0.9, 1.0], taus=[0.5, 1.0, 2.0])
grid.to_csv(config.H5_TRAIN, index=False)
grid.pivot_table(index=["cutoff", "tau"], columns="topology", values=["mean % committed", "spells", "KM median (s)"]).round(2)

KM median (s)                 mean % committed                   \
topology            Full Hierarchy  Star             Full Hierarchy   Star   
cutoff tau                                                                   
0.8    0.5          1.70      1.75  1.71            12.19     13.62  11.06   
       1.0          2.26      1.80  2.28            12.19     13.62  11.06   
       2.0          3.00      3.00  2.28            12.19     13.62  11.06   
0.9    0.5          1.22      1.40  1.38             9.74     11.04   9.34   
       1.0          2.36      1.54  2.61             9.74     11.04   9.34   
       2.0          2.86      2.50  2.46             9.74     11.04   9.34   
1.0    0.5          1.20      1.00  0.85             5.47      6.96   6.79   
       1.0          1.30      1.00  1.09             5.47      6.96   6.79   
       2.0          1.81      1.28  2.31             5.47      6.96   6.79   

           spells                  
topology     Full Hierarchy  Star  
cutoff tau                         
0.8    0.5  208.0     399.0  48.0  
       1.0  185.0     367.0  42.0  
       2.0  145.0     251.0  33.0  
0.9    0.5  220.0     428.0  52.0  
       1.0  159.0     340.0  35.0  
       2.0  117.0     241.0  32.0  
1.0    0.5  178.0     358.0  38.0  
       1.0  161.0     339.0  34.0  
       2.0  121.0     255.0  24.0

In [7]:
grid.set_index(["cutoff", "tau", "topology"])[[a + " %" for a in h5.AFTER]].round(1)

consensus %  relapse %  unresolved %  censored %
cutoff tau topology                                                   
0.8    0.5 Full              24.0       73.1           1.9         1.0
           Hierarchy         16.8       82.0           1.0         0.3
           Star              41.7       58.3           0.0         0.0
       1.0 Full              27.0       69.7           2.2         1.1
           Hierarchy         18.3       80.4           1.1         0.3
           Star              47.6       52.4           0.0         0.0
       2.0 Full              34.5       61.4           2.8         1.4
           Hierarchy         26.7       71.3           1.6         0.4
           Star              60.6       39.4           0.0         0.0
0.9    0.5 Full              19.5       79.1           0.5         0.9
           Hierarchy         14.5       84.3           0.9         0.2
           Star              36.5       63.5           0.0         0.0
       1.0 Full              27.0       71.1           0.6         1.3
           Hierarchy         18.2       80.3           1.2         0.3
           Star              54.3       45.7           0.0         0.0
       2.0 Full              36.8       60.7           0.9         1.7
           Hierarchy         25.7       72.2           1.7         0.4
           Star              59.4       40.6           0.0         0.0
1.0    0.5 Full              19.7       78.7           1.1         0.6
           Hierarchy         13.4       85.2           1.4         0.0
           Star              39.5       60.5           0.0         0.0
       1.0 Full              21.7       76.4           1.2         0.6
           Hierarchy         14.2       84.4           1.5         0.0
           Star              44.1       55.9           0.0         0.0
       2.0 Full              28.9       68.6           1.7         0.8
           Hierarchy         18.8       79.2           2.0         0.0
           Star              62.5       37.5           0.0         0.0

In [8]:
grid[grid["topology"] == "Full"].set_index(["cutoff", "tau"])[
    ["breaker spells (n)", "observed minority", "expected minority (sum k/7)", "(obs - exp) / n", "CI low", "CI high"]].round(3)

breaker spells (n)  observed minority  \
cutoff tau                                          
0.8    0.5               205.0               83.0   
       1.0               182.0               75.0   
       2.0               143.0               65.0   
0.9    0.5               216.0               81.0   
       1.0               155.0               60.0   
       2.0               114.0               52.0   
1.0    0.5               177.0               55.0   
       1.0               160.0               55.0   
       2.0               120.0               47.0   

            expected minority (sum k/7)  (obs - exp) / n  CI low  CI high  
cutoff tau                                                                 
0.8    0.5                       53.286            0.145   0.076    0.218  
       1.0                       47.143            0.153   0.078    0.235  
       2.0                       38.714            0.184   0.105    0.267  
0.9    0.5                       54.143            0.124   0.072    0.173  
       1.0                       38.571            0.138   0.075    0.203  
       2.0                       29.714            0.195   0.120    0.272  
1.0    0.5                       39.000            0.090   0.048    0.138  
       1.0                       36.143            0.118   0.068    0.174  
       2.0                       28.571            0.154   0.095    0.212